# ContentSignal + Wan 2.2 — Google Colab launcher

This notebook mounts your Google Drive, reuses your existing Wan 2.2 GGUF / text encoder / VAE files, starts ComfyUI + FastAPI + the Next.js app, and exposes only the web app port.

In [ ]:
from google.colab import drive, output
drive.mount('/content/drive')


In [ ]:
import os, sys, time, shutil, subprocess, pathlib, requests

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/ContentSignal')
COMFY_DIR = pathlib.Path('/content/ComfyUI')
PROJECT_DIR = pathlib.Path('/content/ContentSignal')
MODEL_ROOT = DRIVE_ROOT / 'video_models'
OUTPUT_DIR = DRIVE_ROOT / 'video_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_model(filename):
    matches = list(MODEL_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(f'Model file not found under {MODEL_ROOT}: {filename}')
    return matches[0]

MODEL_GGUF = find_model('Wan2.2-TI2V-5B-Q4_K_M.gguf')
TEXT_ENCODER = find_model('umt5_xxl_fp8_e4m3fn_scaled.safetensors')
VAE_FILE = find_model('wan2.2_vae.safetensors')
print('Found model files:')
print(MODEL_GGUF)
print(TEXT_ENCODER)
print(VAE_FILE)


In [ ]:
# Get ComfyUI if it is not already present.
if not COMFY_DIR.exists():
    subprocess.run(['git','clone','https://github.com/comfyanonymous/ComfyUI.git',str(COMFY_DIR)],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-r',str(COMFY_DIR/'requirements.txt')],check=True)

# Install the GGUF custom node required by the bundled workflow.
CUSTOM_NODE = COMFY_DIR / 'custom_nodes' / 'ComfyUI-GGUF'
CUSTOM_NODE.parent.mkdir(parents=True, exist_ok=True)
if not CUSTOM_NODE.exists():
    subprocess.run(['git','clone','https://github.com/city96/ComfyUI-GGUF.git',str(CUSTOM_NODE)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(CUSTOM_NODE / 'requirements.txt')],check=True)

# Link your existing Drive model files into ComfyUI's model folders.
def link_model(src, folder):
    target_dir = COMFY_DIR / 'models' / folder
    target_dir.mkdir(parents=True, exist_ok=True)
    target = target_dir / src.name
    if target.is_symlink():
        target.unlink()
    elif target.exists():
        if target.resolve() == src.resolve():
            print('Already linked:', target)
            return
        raise FileExistsError(f'Refusing to replace an existing model file: {target}')
    target.symlink_to(src)
    print('Linked:', target)

link_model(MODEL_GGUF, 'unet')
link_model(TEXT_ENCODER, 'text_encoders')
link_model(VAE_FILE, 'vae')


In [ ]:
# Download/update the ContentSignal source repository.
if not PROJECT_DIR.exists():
    subprocess.run(['git','clone','https://github.com/alza69191-art/ContentSignal.git',str(PROJECT_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(PROJECT_DIR),'pull','--ff-only'],check=True)

subprocess.run([sys.executable,'-m','pip','install','-r',str(PROJECT_DIR/'backend'/'requirements.txt')],check=True)

# Start ComfyUI only if its API is not already responding.
comfy_url = 'http://127.0.0.1:8189'
try:
    requests.get(comfy_url + '/system_stats', timeout=3).raise_for_status()
    print('ComfyUI is already running on port 8189.')
except Exception:
    comfy_log = open('/content/contentsignal-comfyui.log','w')
    subprocess.Popen([sys.executable,'main.py','--listen','0.0.0.0','--port','8189'],cwd=COMFY_DIR,stdout=comfy_log,stderr=subprocess.STDOUT)
    for _ in range(120):
        try:
            requests.get(comfy_url + '/system_stats',timeout=2).raise_for_status()
            print('ComfyUI is ready.')
            break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError('ComfyUI did not start. Check /content/contentsignal-comfyui.log')


In [ ]:
# Start FastAPI. It talks to ComfyUI over localhost; no public ComfyUI port is needed.
backend_env = os.environ.copy()
backend_env.update({
    'COMFYUI_URL': 'http://127.0.0.1:8189',
    'COMFYUI_WORKFLOW_PATH': str(PROJECT_DIR/'backend'/'workflows'/'wan22_ti2v_5b_gguf_api.json'),
    'COMFYUI_PROMPT_NODE_ID': '6',
    'COMFYUI_IMAGE_NODE_ID': '56',
    'COMFYUI_LATENT_NODE_ID': '55',
    'UPLOAD_DIR': str(OUTPUT_DIR/'uploads'),
    'CORS_ORIGINS': 'http://localhost:3000',
})
(OUTPUT_DIR/'uploads').mkdir(parents=True,exist_ok=True)
try:
    requests.get('http://127.0.0.1:8000/health',timeout=2).raise_for_status()
    print('ContentSignal API is already running.')
except Exception:
    api_log = open('/content/contentsignal-api.log','w')
    subprocess.Popen([sys.executable,'-m','uvicorn','app.main:app','--host','0.0.0.0','--port','8000'],cwd=PROJECT_DIR/'backend',env=backend_env,stdout=api_log,stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            requests.get('http://127.0.0.1:8000/health',timeout=2).raise_for_status()
            print('ContentSignal API is ready.')
            break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError('FastAPI did not start. Check /content/contentsignal-api.log')


In [ ]:
# Start the Next.js website. Only port 3000 needs to be exposed to the browser.
if shutil.which('node') is None or shutil.which('npm') is None:
    subprocess.run(['bash','-lc','curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs'],check=True)
node_major = int(subprocess.check_output(['node','-v'],text=True).strip().lstrip('v').split('.')[0])
if node_major < 20:
    raise RuntimeError('Node.js 20+ is required. Restart the runtime after upgrading Node.js.')
frontend_dir = PROJECT_DIR / 'frontend'
subprocess.run(['npm','install'],cwd=frontend_dir,check=True)
web_env = os.environ.copy()
web_env.update({'API_INTERNAL_URL':'http://127.0.0.1:8000','NEXT_PUBLIC_API_URL':''})
web_log = open('/content/contentsignal-web.log','w')
try:
    requests.get('http://127.0.0.1:3000',timeout=2).raise_for_status()
    print('Next.js website is already running.')
except Exception:
    subprocess.Popen(['npm','run','dev','--','--hostname','0.0.0.0','--port','3000'],cwd=frontend_dir,env=web_env,stdout=web_log,stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        requests.get('http://127.0.0.1:3000',timeout=3).raise_for_status()
        print('ContentSignal website is ready.')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Next.js did not start. Check /content/contentsignal-web.log')


In [ ]:
# Open the app inside this notebook. The API and ComfyUI stay internal to Colab.
output.serve_kernel_port_as_iframe(3000, path='/', width='100%', height=800, cache_in_notebook=False)


## Notes

- The bundled workflow starts at 832×480 and 49 frames to reduce VRAM pressure. You can change these values in `backend/workflows/wan22_ti2v_5b_gguf_api.json`.
- Text-to-video and image-to-video are enabled. Video-to-video is not enabled in this initial TI2V-5B integration.
- Keep this Colab runtime active while generating. Closing or disconnecting the runtime stops the model and app.
- If the app does not load, inspect `/content/contentsignal-comfyui.log`, `/content/contentsignal-api.log`, and `/content/contentsignal-web.log`.
